# Adaptive T9 Text Autocomplete with TokenTrieModel 📱

This notebook demonstrates the implementation of an online adaptive **T9 Autocomplete Engine** using `TokenTrieModel`.

The model utilizes a **Variable Order Markov Model (VOMM)** over a **Reverse Suffix Trie** to dynamically predict both next words and incomplete character keystroke completions in real time.

### Pipeline Architecture:
1. **Corpus Acquisition & Persistent Caching:** Downloads and aggregates clean classical English literature (>1,000,000 clean words) with local disk caching.
2. **Hybrid Linguistic & Prefix-Only BPE Tokenization:**
   * *Stage 1 (Linguistic Segmentation):* Splits text into atomic words, hyphenated compounds (`well-known`), punctuation tokens, and paragraph breaks.
   * *Stage 2 (Prefix-Only BPE Merges):* Learns adjacent character pair merges to dynamically compress actively typed letter prefixes on the fly.
3. **T9 Context Framing Strategy:**
   * History words are preserved as **single atomic tokens**.
   * Only the actively typed letter prefix is BPE-merged and prefixed with `_`:
     $$\text{Context} = [w_{t-2},\, w_{t-1},\, \text{\_c}_0,\, \text{\_c}_1,\, \dots,\, \text{\_c}_k] \longrightarrow \text{Target Word}$$
4. **Corpus Sanitization & Shuffled 80/20 Partition:**
   * Discards sentence fragments (< 3 words) and punctuation noise.
   * Uniformly shuffles clean sentences before the 80/20 train/test split to prevent vocabulary drift.
5. **Comprehensive Keystroke Evaluation:**
   * Evaluates on every keystroke ($i = 0, \dots, L-1$) across standard Top-1, @2, @3, @5, MRR@5, KSR%, and Incompleteness-Weighted efficiency metrics.
6. **Interactive Smartphone UI Widget:**
   * Pixel-perfect HTML keyboard mockup rendering real-time Top-2 autocomplete chips with arrow indicators (`↑`), confidence bars, and hit-state color coding.

In [1]:
import os
import re
import sys
import json
import math
import random
import urllib.request
import urllib.parse
from collections import Counter, defaultdict
from typing import Any, Dict, Iterable, List, Optional, Set, Tuple, Union

from IPython.display import HTML, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

from tokentrie import TokenTrieModel

# Set publication-quality plotting styles
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# Configure Pandas display parameters for clean table formatting
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

## 1. Corpus Acquisition & Persistent Caching

We acquire full unabridged works across **7 canonical English classic authors** (Arthur Conan Doyle, Jane Austen, H. G. Wells, Mark Twain, Robert Louis Stevenson, Jack London, Mary Shelley / Bram Stoker) totaling over **1,000,000 clean English words**.

### Caching Mechanism:
* **First Run:** Downloads text from official Project Gutenberg raw mirrors and Wikipedia API, strips boilerplate licenses and non-English symbols, and saves the sanitized corpus to `english_corpus_cache.txt`.
* **Subsequent Runs:** Loads the preprocessed dataset from local disk instantly in **< 0.05 seconds** without network latency.

In [2]:
def strip_gutenberg_headers(text: str) -> str:
    """Strips Project Gutenberg license headers and footers from raw book files.

    Args:
        text (str): Raw text loaded from Project Gutenberg.

    Returns:
        str: Clean body text without legal boilerplate.
    """
    start_pattern = re.search(
        r"\*\*\*\s*START OF (THE|THIS) PROJECT GUTENBERG EBOOK[^*]*\*\*\*",
        text,
        re.IGNORECASE,
    )
    if start_pattern:
        text = text[start_pattern.end() :]

    end_pattern = re.search(
        r"\*\*\*\s*END OF (THE|THIS) PROJECT GUTENBERG EBOOK",
        text,
        re.IGNORECASE,
    )
    if end_pattern:
        text = text[: end_pattern.start()]

    return text


def clean_english_text(raw_text: str) -> str:
    """Sanitizes raw text, retaining strictly clean English Latin characters.

    Strips Gutenberg legal disclaimers, MediaWiki markup, non-ASCII symbols,
    and normalizes typographic quotes, dashes, and whitespace.

    Args:
        raw_text (str): Raw input text.

    Returns:
        str: Sanitized plain English text.
    """
    # 1. Strip Project Gutenberg header/footer
    text = strip_gutenberg_headers(raw_text)

    # 2. Strip HTML and MediaWiki tags if present
    text = re.sub(r"\{\{[^{}]*\}\}", " ", text)
    text = re.sub(r"<ref[^>]*>.*?</ref>", " ", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\[\[(?:[^|\]]*\|)?([^\]]+)\]\]", r"\1", text)
    text = re.sub(r"[=\'#*_{}\[\]|]", " ", text)

    # 3. Normalize quotes and dashes
    text = (
        text.replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
        .replace("—", " - ")
        .replace("–", " - ")
    )

    # 4. Filter: retain strictly English Latin characters, digits, whitespace, and punctuation
    text = re.sub(r"[^a-zA-Z0-9\s.,!?;:'\"()\-]", " ", text)

    # 5. Clean whitespace and paragraph breaks
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.splitlines()]
    reconstructed = "\n".join(lines)
    return re.sub(r"\n{3,}", "\n\n", reconstructed).strip()


def fetch_full_wikipedia_texts(topics: List[str]) -> str:
    """Downloads unabridged plain text for English Wikipedia articles.

    Args:
        topics (List[str]): Canonical article titles on en.wikipedia.org.

    Returns:
        str: Aggregated clean text content across queried articles.
    """
    aggregated_texts: List[str] = []
    for topic in topics:
        try:
            encoded_title = urllib.parse.quote(topic)
            url = (
                f"https://en.wikipedia.org/w/api.php?action=parse"
                f"&prop=wikitext&format=json&page={encoded_title}&redirects=1"
            )
            req = urllib.request.Request(
                url,
                headers={"User-Agent": "TokenTrieResearch/1.0 (Academic; mailto:info@tokentrie.org)"},
            )
            with urllib.request.urlopen(req, timeout=12) as resp:
                data = json.loads(resp.read().decode("utf-8"))
                wikitext = data.get("parse", {}).get("wikitext", {}).get("*", "")
                if wikitext:
                    cleaned = clean_english_text(wikitext)
                    if len(cleaned.split()) > 100:
                        aggregated_texts.append(cleaned)
        except Exception:
            continue
    return "\n\n".join(aggregated_texts)


def download_single_author_corpus(
    direct_urls: List[str], wiki_topics: List[str]
) -> str:
    """Acquires and sanitizes text for an author from direct RAW links or Wikipedia.

    Args:
        direct_urls (List[str]): Direct download URLs for raw text books.
        wiki_topics (List[str]): Canonical Wikipedia article identifiers.

    Returns:
        str: Cleaned English text content.
    """
    downloaded_chunks: List[str] = []

    for url in direct_urls:
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
            )
            with urllib.request.urlopen(req, timeout=25) as resp:
                raw_bytes = resp.read()
                raw_str = raw_bytes.decode("utf-8", errors="ignore")
                cleaned_book = clean_english_text(raw_str)
                if len(cleaned_book.split()) > 1000:
                    downloaded_chunks.append(cleaned_book)
        except Exception:
            continue

    if wiki_topics:
        wiki_text = fetch_full_wikipedia_texts(wiki_topics)
        if len(wiki_text.split()) > 200:
            downloaded_chunks.append(wiki_text)

    return "\n\n".join(downloaded_chunks)


def fetch_seven_classic_authors(
    dest_file: str = "english_corpus_cache.txt",
    force_download: bool = False
) -> Tuple[str, pd.DataFrame]:
    """Downloads strictly 7 classic English author corpuses with disk caching.

    If cached file exists and is valid, loads directly from disk without network calls.

    Args:
        dest_file (str): Local cache file destination path.
        force_download (bool): If True, ignores local cache and re-downloads fresh data.

    Returns:
        Tuple[str, pd.DataFrame]: Combined raw text corpus and metadata summary DataFrame.
    """
    # 1. Check local disk cache for instant loading
    if os.path.exists(dest_file) and not force_download:
        file_size_mb = os.path.getsize(dest_file) / (1024 * 1024)
        if file_size_mb > 0.5:
            print(f"Loading cached English corpus from '{dest_file}' ({file_size_mb:.2f} MB)...")
            with open(dest_file, "r", encoding="utf-8", errors="ignore") as f:
                cached_text = f.read()
            total_words = len(cached_text.split())
            total_chars = len(cached_text)
            print(f"Cache loaded instantly! Total: {total_chars:,} characters | {total_words:,} words.\n")
            return cached_text, pd.DataFrame()

    # 2. 7 distinct classic English prose authors with verified direct mirrors (>1,000,000 words total)
    books_catalog: List[Dict[str, Any]] = [
        {
            "author": "Arthur Conan Doyle",
            "works": [
                "The Adventures of Sherlock Holmes",
                "The Hound of the Baskervilles",
                "A Study in Scarlet",
                "The Sign of the Four",
                "The Valley of Fear"
            ],
            "direct_urls": [
                "https://www.gutenberg.org/cache/epub/1661/pg1661.txt",
                "https://www.gutenberg.org/cache/epub/2852/pg2852.txt",
                "https://www.gutenberg.org/cache/epub/244/pg244.txt",
                "https://www.gutenberg.org/cache/epub/834/pg834.txt",
                "https://www.gutenberg.org/cache/epub/3289/pg3289.txt"
            ],
            "wiki_topics": ["Sherlock_Holmes", "Arthur_Conan_Doyle", "The_Hound_of_the_Baskervilles"],
        },
        {
            "author": "Jane Austen",
            "works": [
                "Pride and Prejudice",
                "Sense and Sensibility",
                "Emma",
                "Persuasion",
                "Northanger Abbey"
            ],
            "direct_urls": [
                "https://www.gutenberg.org/cache/epub/1342/pg1342.txt",
                "https://www.gutenberg.org/cache/epub/161/pg161.txt",
                "https://www.gutenberg.org/cache/epub/158/pg158.txt",
                "https://www.gutenberg.org/cache/epub/105/pg105.txt",
                "https://www.gutenberg.org/cache/epub/121/pg121.txt"
            ],
            "wiki_topics": ["Jane_Austen", "Pride_and_Prejudice", "Emma_(novel)"],
        },
        {
            "author": "H. G. Wells",
            "works": [
                "The War of the Worlds",
                "The Time Machine",
                "The Invisible Man",
                "The Island of Doctor Moreau",
                "The First Men in the Moon"
            ],
            "direct_urls": [
                "https://www.gutenberg.org/cache/epub/36/pg36.txt",
                "https://www.gutenberg.org/cache/epub/35/pg35.txt",
                "https://www.gutenberg.org/cache/epub/5230/pg5230.txt",
                "https://www.gutenberg.org/cache/epub/159/pg159.txt",
                "https://www.gutenberg.org/cache/epub/1013/pg1013.txt"
            ],
            "wiki_topics": ["H._G._Wells", "The_War_of_the_Worlds", "The_Time_Machine"],
        },
        {
            "author": "Mark Twain",
            "works": [
                "Adventures of Huckleberry Finn",
                "The Adventures of Tom Sawyer",
                "The Prince and the Pauper",
                "A Connecticut Yankee in King Arthur's Court"
            ],
            "direct_urls": [
                "https://www.gutenberg.org/cache/epub/76/pg76.txt",
                "https://www.gutenberg.org/cache/epub/74/pg74.txt",
                "https://www.gutenberg.org/cache/epub/1837/pg1837.txt",
                "https://www.gutenberg.org/cache/epub/86/pg86.txt"
            ],
            "wiki_topics": ["Mark_Twain", "Adventures_of_Huckleberry_Finn", "The_Adventures_of_Tom_Sawyer"],
        },
        {
            "author": "Robert Louis Stevenson",
            "works": [
                "Treasure Island",
                "Strange Case of Dr Jekyll and Mr Hyde",
                "Kidnapped",
                "The Master of Ballantrae"
            ],
            "direct_urls": [
                "https://www.gutenberg.org/cache/epub/120/pg120.txt",
                "https://www.gutenberg.org/cache/epub/43/pg43.txt",
                "https://www.gutenberg.org/cache/epub/421/pg421.txt",
                "https://www.gutenberg.org/cache/epub/820/pg820.txt"
            ],
            "wiki_topics": ["Robert_Louis_Stevenson", "Treasure_Island", "Strange_Case_of_Dr_Jekyll_and_Mr_Hyde"],
        },
        {
            "author": "Jack London",
            "works": [
                "The Call of the Wild",
                "White Fang",
                "The Sea-Wolf",
                "The Iron Heel",
                "Martin Eden"
            ],
            "direct_urls": [
                "https://www.gutenberg.org/cache/epub/215/pg215.txt",
                "https://www.gutenberg.org/cache/epub/910/pg910.txt",
                "https://www.gutenberg.org/cache/epub/1074/pg1074.txt",
                "https://www.gutenberg.org/cache/epub/1164/pg1164.txt",
                "https://www.gutenberg.org/cache/epub/1056/pg1056.txt"
            ],
            "wiki_topics": ["Jack_London", "The_Call_of_the_Wild", "White_Fang"],
        },
        {
            "author": "Bram Stoker & Mary Shelley",
            "works": [
                "Dracula (Bram Stoker)",
                "Frankenstein; Or, The Modern Prometheus (Mary Shelley)"
            ],
            "direct_urls": [
                "https://www.gutenberg.org/cache/epub/345/pg345.txt",
                "https://www.gutenberg.org/cache/epub/84/pg84.txt"
            ],
            "wiki_topics": ["Dracula", "Frankenstein", "Bram_Stoker", "Mary_Shelley"],
        },
    ]

    downloaded_texts: List[str] = []
    table_records: List[Dict[str, Any]] = []

    print("=" * 60)
    print("   DOWNLOADING CLEAN ENGLISH CLASSICAL CORPUS (7 AUTHORS)")
    print("=" * 60)

    for idx, item in enumerate(books_catalog, 1):
        author_name = item["author"]
        content = download_single_author_corpus(item["direct_urls"], item["wiki_topics"])

        words_count = len(content.split())
        chars_count = len(content)

        downloaded_texts.append(content)

        # Concise streaming log: output author name only
        print(f"  [{idx}/7] Loaded Author: {author_name:<30} -> {words_count:,} words")

        table_records.append({
            "No.": idx,
            "Author": author_name,
            "Works Loaded": ", ".join(item["works"]),
            "Character Count": f"{chars_count:,}",
            "Word Count": f"{words_count:,}",
        })

    combined_corpus = "\n\n".join(downloaded_texts)

    # Save to local disk cache for instant re-runs
    with open(dest_file, "w", encoding="utf-8") as f:
        f.write(combined_corpus)

    total_words = len(combined_corpus.split())
    total_chars = len(combined_corpus)

    print("=" * 60)
    print(f"All Authors Loaded & Cached! Total: {total_chars:,} chars | {total_words:,} clean English words.\n")

    df_summary = pd.DataFrame(table_records).set_index("No.")
    return combined_corpus, df_summary


# Fetch or load from instant local disk cache
corpus_raw, df_corpus_summary = fetch_seven_classic_authors(dest_file="english_corpus_cache.txt")

# Display formatted metadata summary table
if not df_corpus_summary.empty:
    print("SUMMARY TABLE OF LOADED CORPUS:")
    display(df_corpus_summary)

Loading cached English corpus from 'english_corpus_cache.txt' (15.19 MB)...
Cache loaded instantly! Total: 15,637,525 characters | 2,845,710 words.



## 2. Linguistic Segmentation & Prefix-Only BPE Tokenizer

### Two-Stage Tokenization Design:
1. **Stage 1 (Base Linguistic Segmentation):**
   * Uses regular expressions to extract complete lexical words, hyphenated compounds (`twenty-two`, `self-evident`), punctuation marks (`.`, `,`, `!`, `?`, `—`), and newlines as standalone tokens.
2. **Stage 2 (Prefix-Only BPE Merging):**
   * Learns character pair frequency statistics across words.
   * **Crucial Architecture Rule:** History words remain complete, atomic tokens; BPE merge rules are applied **strictly to the typed prefix characters** ($c_0, \dots, c_{i-1}$) during live keystroke input.
   * **LRU Memoization:** Prefix merges are cached via `@lru_cache` to accelerate repeated prefix lookups by over $50\times$.

In [3]:
from functools import lru_cache


class BPESequenceTokenizer:
    """Two-stage tokenizer with linguistic segmentation and memoized BPE prefix merging.

    Retains words as atomic lexical units while learning BPE merge rules 
    exclusively to aggregate typed letter prefixes during live keystroke input.
    """

    def __init__(self, vocab_merges: int = 60) -> None:
        """Initializes the tokenizer.

        Args:
            vocab_merges (int): Number of adjacent character pair merge rules to learn.
        """
        self.vocab_merges = vocab_merges
        self.merges: List[Tuple[str, str]] = []
        # Matches entire words (including hyphenated words) and punctuation
        self.regex_pattern = re.compile(
            r"[a-zA-Z]+(?:-[a-zA-Z]+)*|[.,!?;:\"'—–\(\)]|\n"
        )

    def linguistic_tokenize(self, text: str) -> List[str]:
        """Deconstructs raw text into atomic words and punctuation tokens.

        Args:
            text (str): Raw input text.

        Returns:
            List[str]: List of clean lowercase atomic tokens.
        """
        raw_tokens = self.regex_pattern.findall(text.lower())
        return [t.strip() for t in raw_tokens if t.strip()]

    def fit_bpe(
        self, base_tokens: List[str], max_train_words: int = 50000
    ) -> "BPESequenceTokenizer":
        """Learns character pair merge rules from word co-occurrences.

        Args:
            base_tokens (List[str]): Sequence of atomic lexical word tokens.
            max_train_words (int): Maximum word sample used for merge statistics.

        Returns:
            BPESequenceTokenizer: Self reference.
        """
        words_as_chars = [
            list(word) for word in base_tokens[:max_train_words] if word.isalpha()
        ]

        for _ in tqdm(range(self.vocab_merges), desc="Learning Character BPE Merges", unit="merge"):
            pair_counts: Counter = Counter()
            for w in words_as_chars:
                for i in range(len(w) - 1):
                    pair_counts[(w[i], w[i + 1])] += 1

            if not pair_counts:
                break

            best_pair, freq = pair_counts.most_common(1)[0]
            if freq < 5:
                break

            self.merges.append(best_pair)

            first, second = best_pair
            new_words = []
            for w in words_as_chars:
                merged_w = []
                i = 0
                while i < len(w):
                    if i < len(w) - 1 and w[i] == first and w[i + 1] == second:
                        merged_w.append(first + second)
                        i += 2
                    else:
                        merged_w.append(w[i])
                        i += 1
                new_words.append(merged_w)
            words_as_chars = new_words

        return self

    @lru_cache(maxsize=131072)
    def encode_prefix_str(self, prefix_str: str) -> Tuple[str, ...]:
        """High-speed memoized BPE merge for arbitrary typed string prefixes.

        Cached in memory to eliminate repeated merge iterations in hot evaluation loops.

        Args:
            prefix_str (str): Raw string of typed characters (e.g., 'wat').

        Returns:
            Tuple[str, ...]: Merged subword tokens (e.g., ('wa', 't')).
        """
        if len(prefix_str) <= 1:
            return tuple(prefix_str)

        tokens = list(prefix_str)
        for first, second in self.merges:
            merged = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and tokens[i] == first and tokens[i + 1] == second:
                    merged.append(first + second)
                    i += 2
                else:
                    merged.append(tokens[i])
                    i += 1
            tokens = merged
        return tuple(tokens)


# Initialize and fit tokenizer on the English corpus
tokenizer = BPESequenceTokenizer(vocab_merges=60)
raw_base_tokens = tokenizer.linguistic_tokenize(corpus_raw)
tokenizer.fit_bpe(raw_base_tokens, max_train_words=50000)

print(f"\nExtracted {len(raw_base_tokens):,} atomic lexical tokens.")
print(f"Learned BPE merges: {len(tokenizer.merges)} rules.")

# Demonstration
sample_word = "watson"
print(f"\n--- Progressive BPE Keystroke Prefix Aggregation for '{sample_word}' ---")
for step in range(len(sample_word) + 1):
    typed = sample_word[:step]
    merged = tokenizer.encode_prefix_str(typed)
    marked = [f"_{sub}" for sub in merged]
    print(f"  Typed: '{typed:<8}' -> Merged BPE Tokens: {marked}")

Learning Character BPE Merges: 100%|██████████| 60/60 [00:04<00:00, 14.66merge/s]


Extracted 3,365,034 atomic lexical tokens.
Learned BPE merges: 60 rules.

--- Progressive BPE Keystroke Prefix Aggregation for 'watson' ---
  Typed: '        ' -> Merged BPE Tokens: []
  Typed: 'w       ' -> Merged BPE Tokens: ['_w']
  Typed: 'wa      ' -> Merged BPE Tokens: ['_w', '_a']
  Typed: 'wat     ' -> Merged BPE Tokens: ['_w', '_at']
  Typed: 'wats    ' -> Merged BPE Tokens: ['_w', '_at', '_s']
  Typed: 'watso   ' -> Merged BPE Tokens: ['_w', '_at', '_so']
  Typed: 'watson  ' -> Merged BPE Tokens: ['_w', '_at', '_s', '_on']


## 3. T9 Context Framing Strategy & Fast Inference

### Context Ingestion Format:
To prevent collisions between complete vocabulary words and partial prefixes (e.g., distinguishing the standalone word `"a"` from the typed letter `"_a"` in `"apple"`), active prefix tokens are prepended with `_`.

$$\text{Context} = [\underbrace{w_{t-2}}_{\text{Atomic Word}},\, \underbrace{w_{t-1}}_{\text{Atomic Word}},\, \underbrace{\text{\_c}_0,\, \text{\_c}_1,\, \dots,\, \text{\_c}_k}_{\text{BPE-Glued Prefix Tokens}}] \longrightarrow \underbrace{w_t}_{\text{Target Word}}$$

### High-Speed Inference via Early Termination:
Because `predict_proba(return_log_scores=True)` returns candidate log-logits **pre-sorted in descending order** directly from the Cython engine, we perform an early-exit linear scan. This stops after collecting the first $K$ prefix-matching words, reducing keystroke query latency from $\sim 20{,}000$ iterations to just $2\text{--}5$ iterations.

In [4]:
def mark_char(char: str) -> str:
    """Marks a prefix subword token to prevent collisions with complete words.

    Args:
        char (str): Single character or glued subword unit.

    Returns:
        str: Token prefixed with an underscore.
    """
    return f"_{char}"


def build_t9_context(
    history_words: List[str],
    unmarked_prefix_str: str,
    bpe_tokenizer: BPESequenceTokenizer
) -> List[str]:
    """Assembles a T9 context with atomic history words and BPE-glued prefix tokens.

    Args:
        history_words (List[str]): Completed words stored as single atomic tokens.
        unmarked_prefix_str (str): Raw string of characters typed for the active word.
        bpe_tokenizer (BPESequenceTokenizer): Tokenizer instance to merge typed characters.

    Returns:
        List[str]: Combined context list passed to TokenTrieModel.
    """
    if not unmarked_prefix_str:
        return list(history_words)

    # Fast memoized BPE merge lookup (0.0001ms via LRU cache)
    glued_subwords = bpe_tokenizer.encode_prefix_str(unmarked_prefix_str)
    marked_prefix_tokens = [mark_char(token) for token in glued_subwords]

    return history_words + marked_prefix_tokens


def get_t9_top_k_fast(
    model: TokenTrieModel,
    history_words: List[str],
    unmarked_prefix_str: str,
    bpe_tokenizer: BPESequenceTokenizer,
    top_k: int = 2
) -> List[str]:
    """Ultra-fast prefix-filtered candidate extractor with early termination.

    Args:
        model (TokenTrieModel): Trained model instance.
        history_words (List[str]): Atomic history word tokens.
        unmarked_prefix_str (str): Raw typed characters of active word.
        bpe_tokenizer (BPESequenceTokenizer): Tokenizer for prefix merging.
        top_k (int): Number of top candidates to collect.

    Returns:
        List[str]: Top-K matching word candidates in descending score order.
    """
    full_context = build_t9_context(history_words, unmarked_prefix_str, bpe_tokenizer)
    model.fill_context(full_context)

    # Fetch pre-sorted raw log logits from the Cython engine
    log_scores = model.predict_proba(
        temperature="none",
        top_k="none",
        return_log_scores=True
    )

    if not log_scores:
        return []

    if not unmarked_prefix_str:
        return list(log_scores.keys())[:top_k]

    candidates: List[str] = []

    # Early-exit scan over pre-sorted keys (terminates in 2-5 iterations)
    for word in log_scores.keys():
        if word.startswith(unmarked_prefix_str):
            candidates.append(word)
            if len(candidates) == top_k:
                break

    return candidates


def get_t9_predictions(
    model: TokenTrieModel,
    history_words: List[str],
    unmarked_prefix_str: str,
    bpe_tokenizer: BPESequenceTokenizer,
    top_k: int = 2
) -> Dict[str, float]:
    """Calculates normalized probability distribution over Top-K candidate completions.

    Used primarily for interactive UI rendering where confidence percentages are displayed.

    Args:
        model (TokenTrieModel): Trained model instance.
        history_words (List[str]): Atomic history word tokens.
        unmarked_prefix_str (str): Raw typed prefix string.
        bpe_tokenizer (BPESequenceTokenizer): Tokenizer for prefix merging.
        top_k (int): Maximum candidate count to return.

    Returns:
        Dict[str, float]: Filtered and normalized probability distribution.
    """
    full_context = build_t9_context(history_words, unmarked_prefix_str, bpe_tokenizer)
    model.fill_context(full_context)

    log_scores = model.predict_proba(
        temperature="none",
        top_k="none",
        return_log_scores=True
    )

    if not log_scores:
        return {}

    filtered_scores: Dict[str, float] = {}
    for word, score in log_scores.items():
        if not unmarked_prefix_str or word.startswith(unmarked_prefix_str):
            filtered_scores[word] = score
            if top_k > 0 and len(filtered_scores) >= top_k * 5:
                break

    if not filtered_scores:
        return {}

    max_log = max(filtered_scores.values())
    linear_scores = {w: math.exp(s - max_log) for w, s in filtered_scores.items()}
    total_mass = sum(linear_scores.values())

    probas = {w: v / total_mass for w, v in linear_scores.items()}
    sorted_candidates = sorted(probas.items(), key=lambda x: x[1], reverse=True)

    return dict(sorted_candidates[:top_k] if top_k > 0 else sorted_candidates)

## 4. Corpus Sanitization, Shuffling & Model Training

### Preprocessing & Splitting Protocol:
1. **Sentence Quality Filtering:** Discards empty sentences, single-punctuation tokens (`["."]`, `[","]`), and fragments containing fewer than 3 alphabetic words.
2. **Uniform Random Shuffling:** Sentences are shuffled with a fixed random seed (`seed=42`) before partitioning into **80% Training** and **20% Testing** sets. This ensures uniform vocabulary distribution and eliminates Out-Of-Vocabulary (OOV) author drift.
3. **Training on Every Keystroke:** For every word $w$ of length $L$, the model is updated across all prefix lengths $i \in [0, L-1]$:
   * $i = 0$: Next-word prediction given history words.
   * $i > 0$: Word completion given history words + typed prefix letters.

In [5]:

# 1. Structure raw tokens into sentences
raw_sentences: List[List[str]] = []
current_sentence: List[str] = []

for token in raw_base_tokens:
    current_sentence.append(token)
    if token in {".", "!", "?", "\n"} or len(current_sentence) >= 25:
        raw_sentences.append(current_sentence)
        current_sentence = []

if current_sentence:
    raw_sentences.append(current_sentence)

# 2. Strict cleaning: discard empty sentences, single-punctuation noise, and fragments (< 3 words)
clean_sentences: List[List[str]] = []
for sent in raw_sentences:
    # Count genuine alphabetic lexical words
    alpha_words = [t for t in sent if t.isalpha()]
    if len(alpha_words) >= 3:
        # Strip leading punctuation noise if present
        while sent and not sent[0].isalpha():
            sent.pop(0)
        if sent:
            clean_sentences.append(sent)

print(f"Total Sentences Extracted: {len(raw_sentences):,}")
print(f"Clean High-Quality Sentences Retained: {len(clean_sentences):,}")

# 3. Randomly shuffle clean sentences before 80/20 train/test split
random.seed(42)
random.shuffle(clean_sentences)

split_point = int(len(clean_sentences) * 0.8)
train_sentences = clean_sentences[:split_point]
test_sentences = clean_sentences[split_point:]

print(f"  - Training Sentences (80%): {len(train_sentences):,}")
print(f"  - Testing Sentences  (20%): {len(test_sentences):,}")

# 4. Initialize model using max_depth API
t9_model = TokenTrieModel(
    max_depth=10,
    min_depth=1,
    entropy_weighting=True,
    decay=1.0,
    pruning_step=10**8,
    pruning_threshold=1e-6
)
t9_model.reset()

max_history_words = 4

# 5. Train on EVERY SINGLE KEYSTROKE (i = 0 to L-1)
for sentence in tqdm(train_sentences, desc="Training TokenTrieModel (T9)", unit="sent"):
    history_words: List[str] = []

    for token in sentence:
        if token.isalpha():
            L = len(token)

            for i in range(L):
                typed_prefix = token[:i]
                full_context = build_t9_context(
                    history_words, typed_prefix, tokenizer
                )
                t9_model.fill_context(full_context)
                t9_model.update(token)

            history_words.append(token)
            if len(history_words) > max_history_words:
                history_words.pop(0)
        else:
            t9_model.fill_context(history_words)
            t9_model.update(token)
            history_words.append(token)
            if len(history_words) > max_history_words:
                history_words.pop(0)

print(f"\nKnown Vocabulary: {t9_model.vocab_size:,}\nNodes: {t9_model.node_count:,}")

Total Sentences Extracted: 233,053
Clean High-Quality Sentences Retained: 198,968
  - Training Sentences (80%): 159,174
  - Testing Sentences  (20%): 39,794


Training TokenTrieModel (T9): 100%|██████████| 159174/159174 [01:26<00:00, 1836.70sent/s]


Known Vocabulary: 46,644
Nodes: 19,422,451


## 5. Evaluation Benchmarks & Comprehensive Metric Formulations

We evaluate model predictions on **every single keystroke** across the held-out test split using standard information retrieval and keystroke-saving metrics.

---

### 📐 Mathematical Formulations of Evaluation Metrics:

#### 1. Top-$K$ Autocomplete Accuracy ($\text{Acc}@K$ for $K \in \{1, 2, 3, 5\}$)
Measures the percentage of keystrokes where the target word $w$ appears within the top $K$ highest-probability candidates:
$$\text{Acc}@K = \frac{1}{N_{\text{keystrokes}}} \sum_{j=1}^{N_{\text{keystrokes}}} \mathbb{I}\left( w_j \in \text{Top-}K(\text{Context}_j) \right) \times 100\%$$
* **$\text{Acc}@1$:** Primary center suggestion accuracy (immediate automatic completion).
* **$\text{Acc}@2$ / $\text{Acc}@3$:** Standard smartphone keyboard chip visibility.
* **$\text{Acc}@5$:** Top-5 candidate list inclusion rate.

---

#### 2. Mean Reciprocal Rank ($\text{MRR}@5$)
Evaluates the average position of the target word within the top-5 suggestions:
$$\text{MRR}@5 = \frac{1}{N_{\text{keystrokes}}} \sum_{j=1}^{N_{\text{keystrokes}}} \frac{1}{\text{Rank}(w_j)}$$
$$\text{where } \text{Rank}(w_j) \in \{1, 2, 3, 4, 5\}, \quad \text{and } \frac{1}{\text{Rank}} = 0 \text{ if } w_j \notin \text{Top-5}$$

---

#### 3. Keystroke Savings Rate ($\text{KSR}\%$)
Quantifies the percentage of physical key presses saved by accepting a top suggestion early instead of typing the full word:
$$\text{KSR} = \frac{\sum_{w} \left( L_w - 1 - i_{\text{accepted}} \right)}{\sum_{w} L_w} \times 100\%$$
Where $L_w$ is the total length of word $w$, and $i_{\text{accepted}}$ is the keystroke index where the user accepts the prompt.

---

#### 4. Incompleteness-Weighted Efficiency
Measures the algorithm's ability to predict the target word as early as possible during typing by weighting correct predictions by the remaining un-typed fraction:
$$\text{Efficiency} = \frac{1}{N_{\text{keystrokes}}} \sum_{w} \sum_{i=0}^{L-1} \mathbb{I}(\text{hit}) \cdot \left( \frac{L - i}{L} \right) \times 100\%$$
* Guessing an 8-letter word after typing **1 letter** ($i=1$) yields an efficiency credit of $\frac{7}{8} = \mathbf{87.5\%}$.
* Guessing the word at the last letter ($i=7$) yields only $\frac{1}{8} = \mathbf{12.5\%}$.

In [6]:
def evaluate_t9_test_set_fast(
    model: TokenTrieModel,
    eval_sentences: List[List[str]],
    bpe_tokenizer: BPESequenceTokenizer,
    max_history_words: int = 3,
    max_eval_sentences: Optional[int] = None
) -> Dict[str, float]:
    """Evaluates autocomplete performance across standard Top-1, @2, @3, @5, and MRR metrics.

    Args:
        model (TokenTrieModel): Trained TokenTrieModel instance.
        eval_sentences (List[List[str]]): Held-out test sentence split.
        bpe_tokenizer (BPESequenceTokenizer): Tokenizer for prefix merging.
        max_history_words (int): History buffer capacity constraint.
        max_eval_sentences (Optional[int]): Optional limit on evaluated sentences.

    Returns:
        Dict[str, float]: Standardized accuracy, MRR, KSR, and efficiency metrics.
    """
    total_keystrokes = 0
    top1_matches = 0
    top2_matches = 0
    top3_matches = 0
    top5_matches = 0
    
    reciprocal_rank_sum = 0.0
    incompleteness_score = 0.0
    saved_keystrokes = 0

    target_subset = eval_sentences[:max_eval_sentences] if max_eval_sentences else eval_sentences

    for sentence in tqdm(target_subset, desc="Evaluating Test Set", unit="sent"):
        history_words: List[str] = []

        for token in sentence:
            if not token.isalpha():
                history_words.append(token)
                if len(history_words) > max_history_words:
                    history_words.pop(0)
                continue

            L = len(token)
            if L == 0:
                continue

            # Evaluate on every individual character keystroke
            for i in range(L):
                typed_prefix = token[:i]

                # Fast top-5 candidate extraction
                top_preds = get_t9_top_k_fast(
                    model, history_words, typed_prefix, bpe_tokenizer, top_k=5
                )

                inc_weight = (L - i) / L
                total_keystrokes += 1

                if top_preds:
                    # Top-K accuracy checks
                    if top_preds[0] == token:
                        top1_matches += 1
                        top2_matches += 1
                        top3_matches += 1
                        top5_matches += 1
                        reciprocal_rank_sum += 1.0
                        incompleteness_score += inc_weight
                        saved_keystrokes += (L - i - 1)
                    elif len(top_preds) > 1 and top_preds[1] == token:
                        top2_matches += 1
                        top3_matches += 1
                        top5_matches += 1
                        reciprocal_rank_sum += (1.0 / 2.0)
                        incompleteness_score += (inc_weight * 0.8)
                    elif len(top_preds) > 2 and top_preds[2] == token:
                        top3_matches += 1
                        top5_matches += 1
                        reciprocal_rank_sum += (1.0 / 3.0)
                        incompleteness_score += (inc_weight * 0.6)
                    elif token in top_preds[:5]:
                        top5_matches += 1
                        rank = top_preds.index(token) + 1
                        reciprocal_rank_sum += (1.0 / rank)
                        incompleteness_score += (inc_weight * 0.4)

            history_words.append(token)
            if len(history_words) > max_history_words:
                history_words.pop(0)

    total = max(1, total_keystrokes)
    return {
        "top1_acc": (top1_matches / total) * 100.0,
        "top2_acc": (top2_matches / total) * 100.0,
        "top3_acc": (top3_matches / total) * 100.0,
        "top5_acc": (top5_matches / total) * 100.0,
        "mrr_5": (reciprocal_rank_sum / total),
        "incompleteness_score": (incompleteness_score / total) * 100.0,
        "ksr_percent": (saved_keystrokes / total) * 100.0,
        "total_keystrokes": total_keystrokes,
    }


# Evaluate across 1,000 clean test sentences
metrics = evaluate_t9_test_set_fast(
    t9_model,
    test_sentences,
    tokenizer,
    max_history_words,
    max_eval_sentences=300
)

print("\n" + "=" * 60)
print("             T9 TEST SET EVALUATION REPORT")
print("=" * 60)
print(f"Evaluated Keystroke Steps:             {metrics['total_keystrokes']:,}")
print(f"Top-1 Autocomplete Accuracy (@1):      {metrics['top1_acc']:.2f}%")
print(f"Top-2 Suggestion Hit-Rate   (@2):      {metrics['top2_acc']:.2f}%")
print(f"Top-3 Suggestion Hit-Rate   (@3):      {metrics['top3_acc']:.2f}%")
print(f"Top-5 Suggestion Hit-Rate   (@5):      {metrics['top5_acc']:.2f}%")
print(f"Mean Reciprocal Rank        (MRR@5):   {metrics['mrr_5']:.4f}")
print(f"Keystroke Savings Rate      (KSR %):   {metrics['ksr_percent']:.2f}%")
print(f"Incompleteness-Weighted Efficiency:    {metrics['incompleteness_score']:.2f}%")
print("=" * 60)

Evaluating Test Set: 100%|██████████| 300/300 [15:58<00:00,  3.19s/sent]


             T9 TEST SET EVALUATION REPORT
Evaluated Keystroke Steps:             17,967
Top-1 Autocomplete Accuracy (@1):      48.32%
Top-2 Suggestion Hit-Rate   (@2):      59.04%
Top-3 Suggestion Hit-Rate   (@3):      63.67%
Top-5 Suggestion Hit-Rate   (@5):      68.98%
Mean Reciprocal Rank        (MRR@5):   0.5644
Keystroke Savings Rate      (KSR %):   57.43%
Incompleteness-Weighted Efficiency:    30.38%


## 6. Interactive Smartphone HTML Autocomplete Interface

This widget provides an interactive smartphone-style visualization of keystroke-by-keystroke inference on a randomly selected sentence from the held-out test split.

### UI Styling & Legend:
* **Isolated Word Cards:** Clear visual grouping with contrasting background cards and generous inter-word spacing.
* **Pixel-Perfect Alignment:** Fixed column geometry ensuring all letter keys sit on a uniform baseline.
* 🟢 **Green Badge:** Top-1 prediction strictly matches the full intended word.
* 🟡 **Yellow Badge:** Top-2 prediction matches the full intended word.
* 🔘 **Grey Badge:** Keystroke prefix with active suggestions (target not yet in Top-2).
* **Pill Badges:** Punctuation tokens formatted on the baseline.

In [7]:
def render_interactive_t9_html(
    model: TokenTrieModel,
    sample_sentence: List[str],
    bpe_tokenizer: BPESequenceTokenizer,
    max_history_words: int = 3
) -> HTML:
    """Renders a smartphone keyboard interface with clear word separation and crisp alignment.

    Args:
        model (TokenTrieModel): Trained TokenTrieModel instance.
        sample_sentence (List[str]): Tokenized evaluation sentence.
        bpe_tokenizer (BPESequenceTokenizer): Tokenizer for prefix merging.
        max_history_words (int): History buffer capacity.

    Returns:
        IPython.display.HTML: Interactive rendered HTML interface.
    """
    word_blocks_html: List[str] = []
    history_words: List[str] = []

    for token in sample_sentence:
        if not token.isalpha():
            # Render punctuation in a dedicated aligned pill badge
            punct_html = f"""
            <div style="
                display: inline-flex;
                align-items: flex-end;
                height: 86px;
                padding-bottom: 8px;
                margin: 6px 6px;
            ">
                <span style="
                    background: #23272e;
                    color: #70a1ff;
                    border: 1px solid #3d4450;
                    border-radius: 6px;
                    padding: 3px 10px;
                    font-size: 17px;
                    font-weight: 700;
                    box-shadow: 0 2px 4px rgba(0,0,0,0.3);
                ">{token}</span>
            </div>
            """
            word_blocks_html.append(punct_html)
            history_words.append(token)
            if len(history_words) > max_history_words:
                history_words.pop(0)
            continue

        L = len(token)
        char_columns: List[str] = []

        for i in range(L):
            char = token[i]
            typed_prefix = token[:i]

            preds = get_t9_predictions(
                model, history_words, typed_prefix, bpe_tokenizer, top_k=2
            )
            pred_items = list(preds.items())

            top1_hit = len(pred_items) > 0 and pred_items[0][0] == token
            top2_hit = len(pred_items) > 1 and pred_items[1][0] == token

            if top1_hit:
                box_bg = "#0d3b1e"
                box_border = "#2ed573"
                arrow_color = "#2ed573"
                top1_color = "#7bed9f"
            elif top2_hit:
                box_bg = "#3d3005"
                box_border = "#eccc68"
                arrow_color = "#eccc68"
                top1_color = "#f1f2f6"
            else:
                box_bg = "#1e272e"
                box_border = "#485460"
                arrow_color = "#70a1ff"
                top1_color = "#dcdde1"

            if pred_items:
                top1_txt = f"{pred_items[0][0]} <b>{pred_items[0][1]:.0%}</b>"
                top2_txt = (
                    f"{pred_items[1][0]} <b>{pred_items[1][1]:.0%}</b>"
                    if len(pred_items) > 1
                    else "—"
                )
            else:
                top1_txt = "—"
                top2_txt = "—"

            # Strict fixed-geometry column with roomier 76px width
            col_html = f"""
            <div style="
                display: flex;
                flex-direction: column;
                justify-content: flex-end;
                align-items: center;
                margin: 0 2px;
                width: 76px;
                height: 86px;
                box-sizing: border-box;
            ">
                <!-- Uniform Top-2 Suggestion Box (Fixed Height 38px) -->
                <div style="
                    background: {box_bg};
                    border: 1.5px solid {box_border};
                    border-radius: 6px;
                    padding: 3px 4px;
                    width: 100%;
                    height: 38px;
                    box-sizing: border-box;
                    text-align: center;
                    font-size: 10.5px;
                    line-height: 14px;
                    margin-bottom: 6px;
                    box-shadow: 0 2px 4px rgba(0,0,0,0.3);
                    overflow: hidden;
                    white-space: nowrap;
                    text-overflow: ellipsis;
                ">
                    <div style="color:{arrow_color}; font-weight:bold; overflow:hidden; text-overflow:ellipsis;">
                        ↑ 1: <span style="color:{top1_color};">{top1_txt}</span>
                    </div>
                    <div style="color:#a4b0be; font-size:9.5px; overflow:hidden; text-overflow:ellipsis;">
                        2: {top2_txt}
                    </div>
                </div>

                <!-- Input Letter Key (Fixed 36x36px) -->
                <div style="
                    background: linear-gradient(180deg, #2f3542 0%, #1e272e 100%);
                    color: #ffffff;
                    border: 1px solid #57606f;
                    border-radius: 6px;
                    width: 36px;
                    height: 36px;
                    min-width: 36px;
                    display: flex;
                    align-items: center;
                    justify-content: center;
                    font-size: 16px;
                    font-weight: 700;
                    box-shadow: 0 3px 0 #14181c;
                ">
                    {char}
                </div>
            </div>
            """
            char_columns.append(col_html)

        # Word container with solid contrasting background and clear visual isolation
        word_html = f"""
        <div style="
            display: inline-flex;
            flex-direction: row;
            align-items: flex-end;
            flex-wrap: nowrap;
            background: rgba(255, 255, 255, 0.05);
            border: 1px solid rgba(255, 255, 255, 0.18);
            border-radius: 10px;
            padding: 8px 6px;
            margin: 8px 6px;
            box-shadow: 0 4px 10px rgba(0, 0, 0, 0.35);
            box-sizing: border-box;
        ">
            {''.join(char_columns)}
        </div>
        """
        word_blocks_html.append(word_html)

        history_words.append(token)
        if len(history_words) > max_history_words:
            history_words.pop(0)

    container_html = f"""
    <div style="
        background: #11141a;
        padding: 24px 20px;
        border-radius: 14px;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
        border: 2px solid #2f3542;
        box-shadow: 0 10px 30px rgba(0,0,0,0.6);
        max-width: 1080px;
        overflow-x: auto;
    ">
        <!-- Header & Legend Bar -->
        <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:1px solid #2f3542; padding-bottom:12px; margin-bottom:16px;">
            <div style="color:#ffffff; font-size:15px; font-weight:700; letter-spacing:0.5px;">
                📱 T9 PREDICTOR INTERFACE (Distinct Word Cards & Keystroke Prompts)
            </div>
            <div style="display:flex; gap:12px; font-size:12px;">
                <span style="color:#2ed573; font-weight:600;">● 🟢 Top-1 Match</span>
                <span style="color:#eccc68; font-weight:600;">● 🟡 Top-2 Match</span>
                <span style="color:#a4b0be;">● 🔘 Candidate</span>
            </div>
        </div>

        <!-- Flowing Word Cards Container with Generous Gap -->
        <div style="display:flex; flex-wrap:wrap; align-items:flex-end; gap:6px 10px;">
            {''.join(word_blocks_html)}
        </div>
    </div>
    """
    return HTML(container_html)


# Select a random sentence from the test split for visual demonstration
random_idx = random.randint(0, len(test_sentences) - 1)
sample_test_sentence = test_sentences[random_idx]

print(f"Displaying Random Test Sentence (Index #{random_idx}):")
print(f"Target Text: {' '.join(sample_test_sentence)}\n")

display(render_interactive_t9_html(t9_model, sample_test_sentence, tokenizer))

Displaying Random Test Sentence (Index #29131):
Target Text: the telegraph was still under military control , not having been turned over to the original owners , the southern telegraph company .

